# 04. FastConformerBPE — обучение BPE-vocab ASR

Self-contained ноутбук: от сырых данных до submission.
Архитектура: Fast Conformer с BPE-256 словарём (~4.2M параметров, subsample_factor=4).
HP Random Search оборачивает весь цикл обучения.

In [ ]:
from gp1.env import setup_environment

paths, device = setup_environment()
print("device:", device)
print("train_csv:", paths.train_csv)
print("dev_csv:", paths.dev_csv)
print("ckpt_root:", paths.ckpt_root)


## Шаг 1. Импорты

In [ ]:
import json
import random
import time

import torch
from torch.utils.data import DataLoader

from gp1.data.dataset import SpokenNumbersDataset
from gp1.data.collate import collate_fn
from gp1.data.audio_aug import AudioAugmenter
from gp1.data.spec_aug import SpecAugmenter
from gp1.data.manifest import records_from_csv
from gp1.features.melbanks import LogMelFilterBanks
from gp1.losses.ctc import CTCLoss
from gp1.models.fast_conformer_bpe import FastConformerBPE
from gp1.text.vocab_bpe import BPEVocab, train_bpe_model
from gp1.text.denormalize import words_to_digits
from gp1.train.trainer import Trainer, TrainerConfig
from gp1.train.optim import build_adamw
from gp1.train.schedulers import build_noam
from gp1.train.checkpoint import load_checkpoint
from gp1.train.metrics import compute_cer, compute_per_speaker_cer
from gp1.decoding.greedy import greedy_decode
from gp1.types import AugConfig

## Шаг 2. Гиперпараметры (FIXED + HP_GRID)

In [ ]:
FIXED = {
    "samplerate": 16000,
    "n_fft": 512,
    "n_mels": 80,
    "hop_length": 160,
    "win_length": 400,
    "max_epochs": 80,
    "grad_accum": 2,
    "subsample_factor": 4,
    "n_heads": 4,
    "ff_ratio": 4,
    "bpe_vocab_size": 256,
}
HP_GRID = {
    "lr":                        [5e-4, 3e-4, 1e-4],
    "weight_decay":              [1e-6, 1e-4],
    "dropout":                   [0.05, 0.1, 0.15],
    "d_model":                   [96, 128],
    "n_blocks":                  [12, 16],
    "conv_kernel":               [9, 15],
    "warmup_steps":              [5000, 10000, 15000],
    "specaug_freq_mask_param":   [15, 20],
    "specaug_time_mask_param":   [25, 35],
    "speed_prob":                [0.5, 1.0],
    "noise_prob":                [0.0, 0.3],
}
N_TRIALS = 6
SEED = 42
print("FIXED:", FIXED)
print("N_TRIALS:", N_TRIALS)


## Шаг 3. Загрузка записей из CSV

In [ ]:
train_records = records_from_csv(paths.train_csv, paths.train_root)
dev_records = records_from_csv(paths.dev_csv, paths.dev_root)
print(f"Train records: {len(train_records)}, Dev records: {len(dev_records)}")


## Шаг 4. BPE Vocabulary

Если файл модели не найден, обучаем SentencePiece BPE на корпусе train-транскрипций.

In [ ]:
BPE_VOCAB_SIZE = FIXED["bpe_vocab_size"]
BPE_MODEL_DIR = paths.ckpt_root.parent / "bpe_models"
BPE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
BPE_MODEL_PATH = BPE_MODEL_DIR / f"bpe_{BPE_VOCAB_SIZE}.model"

if not BPE_MODEL_PATH.exists():
    # Build a plain-text corpus from train transcriptions for SP training.
    corpus_path = BPE_MODEL_DIR / "corpus.txt"
    corpus_path.write_text(
        "\n".join(r.transcription for r in train_records), encoding="utf-8"
    )
    train_bpe_model(
        corpus_path=corpus_path,
        model_prefix=str(BPE_MODEL_DIR / f"bpe_{BPE_VOCAB_SIZE}"),
        vocab_size=BPE_VOCAB_SIZE,
    )
    print(f"BPE model trained: {BPE_MODEL_PATH}")
else:
    print(f"BPE model already exists: {BPE_MODEL_PATH}")

vocab = BPEVocab(BPE_MODEL_PATH)
print(f"BPE vocab_size: {vocab.vocab_size}, blank_id: {vocab.blank_id}")


## Шаг 5. HP Random Search — Training Loop

In [ ]:
SEED = 42
random.seed(SEED)
trial_results = []
run_root = paths.ckpt_root / f"04_fast_conformer_bpe_{int(time.time())}"
run_root.mkdir(parents=True, exist_ok=True)

for trial in range(N_TRIALS):
    hp = {k: random.choice(v) for k, v in HP_GRID.items()}
    print(f"\n=== Trial {trial + 1}/{N_TRIALS} ===")
    print(json.dumps({**FIXED, **hp}, default=str, indent=2))

    aug_cfg = AugConfig(
        speed_prob=hp["speed_prob"],
        noise_prob=hp["noise_prob"],
        seed=SEED + trial,
    )
    train_aug = AudioAugmenter(aug_cfg)
    spec_aug = SpecAugmenter(
        freq_mask_param=hp["specaug_freq_mask_param"],
        num_freq_masks=2,
        time_mask_param=hp["specaug_time_mask_param"],
        num_time_masks=5,
        time_mask_max_ratio=0.05,
        seed=SEED + trial,
    )

    train_ds = SpokenNumbersDataset(
        records=train_records,
        vocab=vocab,
        augmenter=train_aug,
        target_samplerate=FIXED["samplerate"],
    )
    dev_ds = SpokenNumbersDataset(
        records=dev_records,
        vocab=vocab,
        augmenter=None,
        target_samplerate=FIXED["samplerate"],
    )
    train_loader = DataLoader(
        train_ds,
        batch_size=16,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=2,
    )
    dev_loader = DataLoader(
        dev_ds,
        batch_size=16,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=2,
    )

    model = FastConformerBPE(
        vocab_size=vocab.vocab_size,
        d_model=hp["d_model"],
        n_blocks=hp["n_blocks"],
        n_heads=FIXED["n_heads"],
        ff_ratio=FIXED["ff_ratio"],
        conv_kernel=hp["conv_kernel"],
        dropout=hp["dropout"],
        subsample_factor=FIXED["subsample_factor"],
    ).to(device)

    ctc = CTCLoss(blank_id=vocab.blank_id)
    optimizer = build_adamw(
        model.parameters(),
        lr=hp["lr"],
        weight_decay=hp["weight_decay"],
    )
    scheduler = build_noam(
        optimizer,
        d_model=hp["d_model"],
        warmup_steps=hp["warmup_steps"],
    )

    trial_dir = run_root / f"trial_{trial:02d}"
    cfg = TrainerConfig(
        max_epochs=FIXED["max_epochs"],
        grad_accum=FIXED["grad_accum"],
        fp16_autocast=True,
        val_every_n_epochs=1,
        early_stop_patience=10,
        early_stop_metric="max_speaker_cer",
        ckpt_dir=trial_dir,
    )
    trainer = Trainer(
        model=model,
        ctc_loss=ctc,
        optimizer=optimizer,
        scheduler=scheduler,
        train_loader=train_loader,
        val_loader=dev_loader,
        vocab=vocab,
        config=cfg,
        device=device,
        audio_cfg={
            "n_fft": FIXED["n_fft"],
            "n_mels": FIXED["n_mels"],
            "hop_length": FIXED["hop_length"],
            "win_length": FIXED["win_length"],
            "samplerate": FIXED["samplerate"],
        },
        spec_augmenter=spec_aug,
    )
    result = trainer.fit()
    best_ckpt = result["best_ckpt_path"]
    trial_results.append({"trial": trial, "hp": hp, **result})

    if best_ckpt is not None:
        with open(trial_dir / "meta.json", "w") as f:
            json.dump(
                {
                    "arch": "fast_conformer_bpe",
                    "hparams": {**FIXED, **hp},
                    "val_cer": result["best_val_cer"],
                    "trial": trial,
                },
                f,
                indent=2,
                default=str,
            )

print("\nHP search complete.")


## Шаг 6. Сводный отчёт трайлов

In [ ]:
trial_results_sorted = sorted(trial_results, key=lambda r: r["best_val_cer"])
print(f"{'trial':>5} {'best_val_cer':>12} {'lr':>8} {'dropout':>8} {'d_model':>8} {'n_blocks':>9}")
print("-" * 57)
for r in trial_results_sorted:
    hp = r["hp"]
    print(
        f"{r['trial']:>5}"
        f" {r['best_val_cer']:>12.4f}"
        f" {hp['lr']:>8.5f}"
        f" {hp['dropout']:>8.4f}"
        f" {hp['d_model']:>8}"
        f" {hp['n_blocks']:>9}"
    )

## Шаг 7. Валидация лучшей модели на dev + greedy predictions

In [ ]:
best_result = trial_results_sorted[0]
best_ckpt = best_result["best_ckpt_path"]
best_hp = best_result["hp"]
print("Best trial val_cer:", best_result["best_val_cer"], "ckpt:", best_ckpt)

model = FastConformerBPE(
    vocab_size=vocab.vocab_size,
    d_model=best_hp["d_model"],
    n_blocks=best_hp["n_blocks"],
    n_heads=FIXED["n_heads"],
    ff_ratio=FIXED["ff_ratio"],
    conv_kernel=best_hp["conv_kernel"],
    dropout=best_hp["dropout"],
    subsample_factor=FIXED["subsample_factor"],
).to(device)
meta = load_checkpoint(best_ckpt, model)
model.eval()

mel_extractor = LogMelFilterBanks(
    n_fft=FIXED["n_fft"],
    samplerate=FIXED["samplerate"],
    hop_length=FIXED["hop_length"],
    win_length=FIXED["win_length"],
    n_mels=FIXED["n_mels"],
).to(device)

dev_ds_eval = SpokenNumbersDataset(
    records=dev_records, vocab=vocab, augmenter=None, target_samplerate=FIXED["samplerate"]
)
dev_loader_eval = DataLoader(
    dev_ds_eval, batch_size=16, shuffle=False, collate_fn=collate_fn, num_workers=2
)

predictions, refs, spks = [], [], []
with torch.no_grad():
    for batch in dev_loader_eval:
        audio = batch.audio.to(device)
        audio_lengths = batch.audio_lengths.to(device)
        mel = mel_extractor(audio)
        mel_lengths = (audio_lengths // FIXED["hop_length"] + 1).clamp(max=mel.size(-1)).long()
        out = model(mel, mel_lengths)
        hyps = greedy_decode(out.log_probs, out.output_lengths, vocab)
        predictions.extend(hyps)
        refs.extend(batch.transcriptions)
        spks.extend(batch.spk_ids)

# BPE vocab decodes piece ids -> text; then normalize to digit strings.
digit_preds = [words_to_digits(h) for h in predictions]
cer = compute_cer(refs, digit_preds)
per_spk = compute_per_speaker_cer(refs, digit_preds, spks)
print(f"Dev CER (greedy BPE): {cer:.4f}")
print("Per-speaker CER:", per_spk)

## Шаг 8. Beam search + KenLM

Beam search + KenLM не поддерживается для BPE-vocab в текущей реализации.
`gp1.decoding.beam_pyctc` ориентирован на char-vocab.
Используйте greedy decode (шаг 7) или реализуйте BPE-совместимый beam decoder отдельно.

## Шаг 9. Submission (если test данные доступны)

In [ ]:
if paths.test_root is not None:
    from gp1.submit.inference_utils import build_test_dataloader, write_submission

    test_records = records_from_csv(paths.test_root / "test.csv", paths.test_root)
    test_loader = build_test_dataloader(test_records, vocab=vocab)

    all_hyps = []
    model.eval()
    with torch.no_grad():
        for batch in test_loader:
            audio = batch.audio.to(device)
            audio_lengths = batch.audio_lengths.to(device)
            mel = mel_extractor(audio)
            mel_lengths = (
                (audio_lengths // FIXED["hop_length"] + 1).clamp(max=mel.size(-1)).long()
            )
            out = model(mel, mel_lengths)
            hyps = greedy_decode(out.log_probs, out.output_lengths, vocab)
            all_hyps.extend(hyps)

    # BPE vocab already decodes to text; normalize to digit strings.
    pairs = [
        (rec.audio_path.name, words_to_digits(h))
        for rec, h in zip(test_records, all_hyps)
    ]
    submission_path = run_root / "submission.csv"
    write_submission(pairs, submission_path)
    print("Submission saved:", submission_path)
else:
    print("No test_root — skipping submission.")
